<a href="https://colab.research.google.com/github/icarovazquez/airline-data-mesh/blob/notebook-01-build-mesh/notebooks/01_build_data_mesh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook I: Build the Datalake and check the December 2022 data

This notebook builds the first two divisions of our fictional airline’s data mesh from public data, then uses them to ask one question: when an airline cancelled thousands of flights in late December 2022, was the weather the cause?

Raw files go into a lake (Google Drive). Each division’s data is then published to a warehouse (BigQuery), one dataset per division, and we query the warehouse to look for the answer.

Before you start: create a free Google Cloud project and copy its project ID. You’ll paste it into Cell 1.

**Step 1:** Get Ready

Here we connect the notebook to two places:


*   Google Drive is our datalake stand-in so the raw files are saved there
*   BigQuery is our “warehouse.” You’ll be asked to sign in with your Google account. Before running it, replace YOUR_GCP_PROJECT_ID with the ID of the Google Cloud project you created. This is also where you choose which month of flights to load and which airline to follow.



In [2]:
#setup
! pip -q install pandas pyarrow requests

from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

from pathlib import Path
from google.cloud import bigquery

PROJECT="airline-data-mesh"
bq = bigquery.Client(project=PROJECT)
ROOT = Path("/content/drive/MyDrive/airline-data-mesh/lake")
ROOT.mkdir(parents=True, exist_ok=True)
MONTHS = [(2022, 12)]
CARRIER = 'WN'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Step 2**: Download the raw flights

The U.S. government publishes a record of every domestic flight, one row per flight, with its scheduled time, delay and whether it was cancelled. This cell downloads one month of that file and saves it, untouched, in the lake. If the file is already there from an earlier run, it skips the download.

In [3]:
#download the BTS file to drive

import requests, zipfile, io

raw = ROOT / '_raw'; raw.mkdir(exist_ok=True)
for y, m in MONTHS:
    f = raw / f'bts_{y}_{m:02d}.csv'
    if f.exists():
        continue
    url = ('https://transtats.bts.gov/PREZIP/'
           f'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip')
    r = requests.get(url, timeout=600); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        f.write_bytes(z.read(z.namelist()[0]))
    print('saved', f.name, f.stat().st_size // 1_000_000, 'MB')

saved bts_2022_12.csv 251 MB


**Step 3:** Build the flights data product

Keep only our airline. The file covers every airline, and we only need one. This cell keeps that airline’s flights, keeps the columns we care about, and gives them clear names. It then saves the result in a compact format called Parquet. This is the first “data product”: the flights, cleaned up and ready to share.

In [4]:
#Network Operations Product -> Parquet (landing format)

import pandas as pd

RENAME = {
    'FlightDate': 'flight_date', 'Reporting_Airline': 'carrier',
    'Tail_Number': 'tail_number', 'Flight_Number_Reporting_Airline': 'flight_number',
    'Origin': 'origin', 'Dest': 'dest', 'CRSDepTime': 'crs_dep_time',
    'DepDelay': 'dep_delay_min', 'Cancelled': 'cancelled',
    'CancellationCode': 'cancel_code', 'Diverted': 'diverted',
    'CarrierDelay': 'carrier_delay_min', 'WeatherDelay': 'weather_delay_min',
    'NASDelay': 'nas_delay_min', 'SecurityDelay': 'security_delay_min',
    'LateAircraftDelay': 'late_aircraft_delay_min'}
parts = []
for f in sorted(raw.glob('bts_*.csv')):
    df = pd.read_csv(f, usecols=list(RENAME), low_memory=False)
    parts.append(df[df.Reporting_Airline == CARRIER])
fl = pd.concat(parts, ignore_index=True).rename(columns=RENAME)
fl['flight_date'] = pd.to_datetime(fl['flight_date']).dt.date
out = ROOT / 'flight_ops'; out.mkdir(exist_ok=True)
fl.to_parquet(out / 'flights.parquet', index=False)
print(len(fl), 'flights,', int(fl.cancelled.sum()), 'cancelled')

116229 flights, 17000 cancelled


**Step 4** Build the weather data product

Download the weather.

Airport weather stations record conditions every hour: temperature, visibility, wind and anything like snow or thunderstorms.

This cell downloads that weather for the 25 busiest airports in our data and saves it the same way. It asks for all 25 airports in one request, because the weather service blocks us if we ask too often. This is our second data product.

In [6]:
# downloading weather data product from (Iowa Mesonet ASOS) for the 25 busiest airports -> Parquet

import time

airports = fl.origin.value_counts().head(25).index.tolist()
p = dict(station=airports, data=['tmpf', 'vsby', 'sknt', 'wxcodes'],
         year1=2022, month1=12, day1=1, year2=2023, month2=1, day2=1,
         tz='Etc/UTC', format='onlycomma', latlon='no', missing='M',
         trace='T', direct='no', report_type=[3, 4])
for attempt in range(5):
    r = requests.get('https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py',
                     params=p, timeout=300)
    if r.status_code == 429:
        wait = 30 * (attempt + 1)
        print(f'rate limited, waiting {wait}s')
        time.sleep(wait)
        continue
    r.raise_for_status()
    break
else:
    raise RuntimeError('still rate limited; wait a few minutes and rerun this cell')
wx = pd.read_csv(io.StringIO(r.text), na_values=['M'])
wx['valid'] = pd.to_datetime(wx['valid'])
wxdir = ROOT / 'weather'; wxdir.mkdir(exist_ok=True)
wx.to_parquet(wxdir / 'observations.parquet', index=False,
              coerce_timestamps='us', allow_truncated_timestamps=True)
print(len(wx), 'observations,', wx.station.nunique(), 'airports')

22667 observations, 25 airports


**Step 5**: Publish to BigQuery

Move the data into the warehouse.

Right now the data is in files.

This cell loads it into BigQuery, with the flights in one dataset and the weather in another. That mirrors a real company, where each department owns its own data. It also counts the flights it loaded, so you can see that it worked.

In [7]:
# publish each division's product to BigQuery, each division will have its own dataset

def publish(dataset, table, parquet_path):
    bq.create_dataset(bigquery.Dataset(f'{PROJECT}.{dataset}'), exists_ok=True)
    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition='WRITE_TRUNCATE')
    with open(parquet_path, 'rb') as f:
        bq.load_table_from_file(f, f'{PROJECT}.{dataset}.{table}',
                                job_config=cfg).result()

publish('network_ops', 'flights', out / 'flights.parquet')
publish('weather', 'observations', wxdir / 'observations.parquet')
print(bq.query(f'SELECT count(*) AS n FROM `{PROJECT}.network_ops.flights`')
      .to_dataframe())

        n
0  116229


**Step 6**: Check the data, by airport

Did cancellations follow the weather?

For the last eleven days of December, this cell compares each airport’s cancellations with its weather.

It prints one row per airport, with the share of flights cancelled and the share of the time the weather was bad.

The second table lists airports where the weather was fine but lots of flights were still cancelled. Those are the ones that point to something other than weather.

In [10]:
#the Case A check, in BigQuery

sql = f"""
WITH wx AS (
  SELECT station, DATE(valid) AS day,
    AVG(CAST(REGEXP_CONTAINS(IFNULL(wxcodes, ''), r'SN|FZ|TS|PL|GR|IC')
             OR vsby < 3 OR sknt >= 35 AS INT64)) AS bad_wx_share
  FROM `{PROJECT}.weather.observations`
  GROUP BY station, day),
fl AS (
  SELECT origin, flight_date AS day, COUNT(*) AS flights,
    SUM(cancelled) AS cancelled,
    COUNTIF(cancel_code = 'A') AS code_a,
    COUNTIF(cancel_code = 'B') AS code_b,
    COUNTIF(cancel_code = 'C') AS code_c
  FROM `{PROJECT}.network_ops.flights`
  WHERE flight_date BETWEEN '2022-12-21' AND '2022-12-31'
  GROUP BY origin, day)
SELECT fl.origin, SUM(flights) AS flights, SUM(cancelled) AS cancelled,
  ROUND(SUM(cancelled) / SUM(flights), 2) AS cancel_rate,
  SUM(code_a) AS code_a, SUM(code_b) AS code_b, SUM(code_c) AS code_c,
  ROUND(AVG(bad_wx_share), 2) AS bad_wx_share
FROM fl JOIN wx ON wx.station = fl.origin AND wx.day = fl.day
GROUP BY fl.origin ORDER BY cancel_rate DESC"""
t = bq.query(sql).to_dataframe()
print(t.to_string(index=False))
print('\nClear weather but heavy cancellations:')
print(t[(t.bad_wx_share < 0.05) & (t.cancel_rate > 0.2)].to_string(index=False))

origin  flights  cancelled  cancel_rate  code_a  code_b  code_c  bad_wx_share
   SNA      592      293.0         0.49     272      18       3          0.06
   MCI      588      274.0         0.47     224      49       1          0.12
   SAN     1127      530.0         0.47     471      51       8          0.13
   DEN     2794     1293.0         0.46     943     347       3          0.14
   BNA     1387      642.0         0.46     524     118       0          0.11
   SMF      946      433.0         0.46     399      31       3          0.53
   BUR      552      249.0         0.45     239      10       0          0.04
   SJC     1003      444.0         0.44     414      26       4          0.05
   FLL      516      222.0         0.43     207      14       1          0.01
   MDW     2200      955.0         0.43     670     281       4          0.23
   DCA      464      196.0         0.42     163      33       0          0.08
   MSY      574      238.0         0.41     223      15       0 

**Step 7**: Check day by day

Watch it unfold.

Same question, but for each day and for five airports.

We’re looking for two things: airports where cancellations keep rising even though the weather is clear, and days when every airport gets worse together, whatever its local weather.

In [11]:
# day by day at a few airports, before and after the storm
sql = f"""
WITH wx AS (
  SELECT station, DATE(valid) AS day,
    AVG(CAST(REGEXP_CONTAINS(IFNULL(wxcodes, ''), r'SN|FZ|TS|PL|GR|IC')
             OR vsby < 3 OR sknt >= 35 AS INT64)) AS bad_wx_share
  FROM `{PROJECT}.weather.observations`
  GROUP BY station, day),
fl AS (
  SELECT origin, flight_date AS day, COUNT(*) AS flights, SUM(cancelled) AS cancelled
  FROM `{PROJECT}.network_ops.flights`
  WHERE origin IN ('LAS', 'DAL', 'PHX', 'DEN', 'MDW')
    AND flight_date BETWEEN '2022-12-18' AND '2022-12-31'
  GROUP BY origin, day)
SELECT fl.origin, fl.day, flights, cancelled,
  ROUND(cancelled / flights, 2) AS cancel_rate,
  ROUND(bad_wx_share, 2) AS bad_wx_share
FROM fl JOIN wx ON wx.station = fl.origin AND wx.day = fl.day
ORDER BY fl.origin, fl.day"""
d = bq.query(sql).to_dataframe()
print(d.pivot(index='day', columns='origin',
              values=['cancel_rate', 'bad_wx_share']).to_string())

           cancel_rate                         bad_wx_share                       
origin             DAL   DEN   LAS   MDW   PHX          DAL   DEN  LAS   MDW   PHX
day                                                                               
2022-12-18        0.00  0.00  0.00  0.00  0.00         0.00  0.00  0.0  0.62  0.00
2022-12-19        0.00  0.01  0.00  0.00  0.00         0.00  0.00  0.0  0.00  0.00
2022-12-20        0.01  0.00  0.00  0.00  0.00         0.16  0.00  0.0  0.00  0.00
2022-12-21        0.04  0.28  0.02  0.02  0.05         0.00  0.14  0.0  0.00  0.00
2022-12-22        0.19  0.75  0.16  0.55  0.17         0.16  0.71  0.0  0.50  0.00
2022-12-23        0.26  0.44  0.23  0.73  0.23         0.00  0.00  0.0  1.00  0.00
2022-12-24        0.34  0.45  0.37  0.51  0.34         0.00  0.00  0.0  0.50  0.00
2022-12-25        0.41  0.57  0.50  0.53  0.44         0.00  0.00  0.0  0.00  0.00
2022-12-26        0.74  0.81  0.82  0.74  0.73         0.00  0.26  0.0  0.52  0.00
2022

**Step 8**: Schedule and Aircraft

Did the airline have the planes?

This cell counts, for each day across the whole airline, how many flights were scheduled and how many different aircraft actually flew.

If the schedule stays the same but far fewer aircraft fly, the airline didn’t plan fewer flights. It lost the ability to fly them.

In [12]:
# did the schedule itself shrink? Flights and aircraft per day
sql = f"""
SELECT flight_date AS day, COUNT(*) AS scheduled, SUM(cancelled) AS cancelled,
  COUNT(DISTINCT IF(cancelled = 0, tail_number, NULL)) AS tails_flying
FROM `{PROJECT}.network_ops.flights`
WHERE flight_date BETWEEN '2022-12-18' AND '2022-12-31'
GROUP BY day ORDER BY day"""
print(bq.query(sql).to_dataframe().to_string(index=False))

       day  scheduled  cancelled  tails_flying
2022-12-18       3917        2.0           692
2022-12-19       3920       29.0           701
2022-12-20       3920       10.0           696
2022-12-21       3918      178.0           698
2022-12-22       3918     1026.0           682
2022-12-23       3918     1445.0           654
2022-12-24       3252     1420.0           612
2022-12-25       3590     1854.0           593
2022-12-26       3918     3037.0           413
2022-12-27       3918     2731.0           458
2022-12-28       3918     2525.0           488
2022-12-29       3918     2397.0           532
2022-12-30       3918       44.0           702
2022-12-31       3256       34.0           692
